# GO-topic spatial enrichment in the adult brain (AHBA)

The same regional-enrichment analysis as notebook 04_01, run on the three GO topics
(**SYN**, **GR**, **MORPH**) that partition the ASD risk set, plus the tests that ask what
kind of spatial pattern each topic map carries and whether that pattern converges with
independently reported cortical transcriptomic dysregulation.

> **Kernel:** the `spin` conda env (`Python (spin)`). **Run from this directory.**

**Sections**

1. Regional enrichment per topic and the cortical surface maps
2. Spin test read-back and the A–P spin panels (Fig 2F)
3. Map pattern tests — non-uniformity, and whether the three maps share one pattern
4. Occipital and gradient sensitivity (one supplementary figure)
5. Convergence with Gandal et al. 2022 cortical transcriptomic attenuation

Topic assignments come from the GO topic-modelling analysis
(`outputs/analysis/03_module_enrichments/03_02_gene-topic-assignment.csv`). Each topic is
tested against the **same** brain-expressed background as the full ASD set, so the topic
maps are directly comparable to the all-ASD map from notebook 01.

## 0) Setup

In [ ]:
import os, glob, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import spatial_helpers as sh
warnings.filterwarnings("ignore")

PROJ = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))  # repo root (run from this directory)
RESULTS  = f"{PROJ}/outputs/analysis/04_brain_spatial_enrichment"
CACHE    = f"{RESULTS}/spin_cache"
FIG_MAIN = f"{PROJ}/outputs/figures/Fig2"          # topic maps are main-text panel Fig 2F
FIG_SUPP = f"{PROJ}/outputs/figures/supplement"
for d in (RESULTS, FIG_MAIN, FIG_SUPP):
    os.makedirs(d, exist_ok=True)

plt.rcParams.update({"font.size": 7, "axes.titlesize": 8, "axes.labelsize": 7,
                     "xtick.labelsize": 6, "ytick.labelsize": 6,
                     "pdf.fonttype": 42, "ps.fonttype": 42, "mathtext.default": "regular",
                     "font.family": "sans-serif",
                     "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"]})
P_SPIN, P_PERM = r"$P_{\mathrm{spin}}$", r"$P_{\mathrm{perm}}$"

TOPICS = ["GR", "SYN", "MORPH"]
TCOL = sh.TOPIC_COLORS
latest = lambda pattern: sorted(glob.glob(pattern))[-1]

topic_df = pd.read_csv(f"{PROJ}/outputs/analysis/03_module_enrichments/"
                       "03_02_gene-topic-assignment.csv")
GENE_SETS = {t: topic_df.loc[topic_df.GO_topic == t, "gene_name"].unique().tolist() for t in TOPICS}

expr, info = sh.load_ahba()
background = sh.brain_expressed_background(expr)
Z = sh.zscore_expression(expr, background)
dk = sh.dk_region_order()

for t in TOPICS:
    print(f"{t:6s}: {len(GENE_SETS[t]):3d} genes "
          f"({sum(g in background for g in GENE_SETS[t])} brain-expressed)")

## 1) Regional enrichment per topic

Identical procedure to notebook 04_01: per-gene z-score across regions, mean z across the
topic's genes, compared to 10,000 size-matched random sets from the same brain-expressed
background. The random-set size matches each topic's gene count, so the null is
size-appropriate despite the topics differing in size.

In [ ]:
topic_enr = {}
for t in TOPICS:
    enr = sh.regional_enrichment(Z, info, GENE_SETS[t], n_perm=10000, seed=0)
    enr.to_csv(f"{RESULTS}/04_02_{t}_regional_enrichment.csv", index=False)
    topic_enr[t] = enr
    print(f"{t:6s}: FDR<0.05 in {(enr.fdr < 0.05).sum()}/{len(enr)} regions")

# Shared symmetric colour limit across topics, scaled to cortex only, so white stays at 0
# and intensity is comparable between the three maps.
topic_vmax = max(sh.cortical_vmax(topic_enr[t]) for t in TOPICS)
topic_map = {t: np.array([dict(zip(topic_enr[t].region, topic_enr[t].z_vs_bg)).get(r, np.nan)
                          for r in dk]) for t in TOPICS}
print(f"\nshared vmax = {topic_vmax:.2f}")

## 2) Spin test

Run the spin test on each topic map. The spins and gradient nulls are cached and shared
across topics, so only the first run pays the projection cost:

```
./submit_spin_tests.sh
```

or locally:

```
for T in SYN GR MORPH; do
  python run_spin_test.py --n-spin 5000 --prefix 04_02_${T}_ \
      --asd-map $OUT/04_02_${T}_regional_enrichment.csv --asd-cols z_vs_bg
done
```

In [ ]:
SIG_COL, SIG_CUT = "q", 0.05          # switch to ("p_fwe", 0.05) for max-statistic FWE

spin_corr, sig_regions, occ_focal = {}, {}, {}
for t in TOPICS:
    spin_corr[t] = pd.read_csv(latest(f"{RESULTS}/04_02_{t}_spin_results_41k_*.csv")).set_index("C")
    pr = pd.read_csv(latest(f"{RESULTS}/04_02_{t}_cortex_perregion_41k_*.csv"))
    sig_regions[t] = set(pr.loc[pr[SIG_COL] < SIG_CUT, "C"])
    occ_focal[t] = pd.read_csv(latest(f"{RESULTS}/04_02_{t}_occipital_focal_41k_*.csv")).iloc[0]

summary = pd.DataFrame([{"topic": t,
                         "rho_AP": -spin_corr[t].loc["AP", "enrich_z"],   # posterior-positive
                         "p_AP": spin_corr[t].loc["AP", "p"],
                         "occipital_z": occ_focal[t]["enrich_z"],
                         "p_occipital": occ_focal[t]["p"],
                         f"n_regions_{SIG_COL}<{SIG_CUT}": len(sig_regions[t])} for t in TOPICS])
print(summary.round(4).to_string(index=False))

## 3) Main-text figure elements (Fig 2F)

Same export scheme as notebook 04_01 — brain maps as raster-embedded PDFs, colorbars and the
spin panels as true vector. Surface density is `spatial_helpers.DENSITY`, currently
`"fsaverage"` (164k). **Set `sh.DENSITY = "fsaverage5"` for a fast 10k draft.**

The A–P **scatter** panels are main-text for the all-ASD map only (notebook 04_01); for the
topics the spin-null panel in §3.2 carries the A–P result instead.

### 3.1 Cortical surface maps + colorbars

In [ ]:
for t in TOPICS:
    z_by_region = dict(zip(topic_enr[t].region, topic_enr[t].z_vs_bg))
    cmap = sh.diverging_cmap(TCOL[t])          # black -> white -> topic colour

    fig = sh.plot_dk_surface(z_by_region, cmap, topic_vmax, outline=sig_regions[t], bare=True)
    fig.savefig(f"{FIG_MAIN}/Fig2F_{t}_map_clean.png", dpi=300, bbox_inches="tight", pad_inches=0)
    plt.show()

    img = plt.imread(f"{FIG_MAIN}/Fig2F_{t}_map_clean.png")
    f = plt.figure(figsize=(2.2, 2.2 / (img.shape[1] / img.shape[0])))
    f.add_axes([0, 0, 1, 1]).imshow(img); f.axes[0].axis("off")
    f.savefig(f"{FIG_MAIN}/Fig2F_{t}_brain.pdf", dpi=300); plt.close(f)

    leg = sh.colorbar_panel(cmap, topic_vmax, label="Z")
    leg.savefig(f"{FIG_MAIN}/Fig2F_{t}_legend.pdf", bbox_inches="tight")
    leg.savefig(f"{FIG_MAIN}/Fig2F_{t}_legend.png", dpi=600, bbox_inches="tight"); plt.close(leg)
print("saved brain + legend for", TOPICS)

### 3.2 A–P spin-null panels

Each topic map correlated with 5,000 spun copies of the A–P axis. **Each topic gets its
own null** — the null's width depends on that map's spatial smoothness, so GR's is
slightly wider than MORPH's — which is why they are stacked as a column rather than
sharing one distribution. Plotted posterior-positive (+ = posterior-aligned); `*` marks
spin p < 0.05. Observed ρ come from the cluster run and the reconstructed p reproduces it.

In [ ]:
spun_ap = np.load(latest(f"{CACHE}/nulls_41k_n*_seed0_4comp.npy"))[:, 3, :]   # AP = component 3
ap_null = {t: -np.array([sh.masked_spearman(topic_map[t], spun_ap[:, i])
                         for i in range(spun_ap.shape[1])]) for t in TOPICS}
ap_res = pd.DataFrame([{"topic": t,
                        "rho_AP": -spin_corr[t].loc["AP", "enrich_z"],
                        "P_spin": sh.emp_p(-spin_corr[t].loc["AP", "enrich_z"], ap_null[t]),
                        "P_spin_cluster": spin_corr[t].loc["AP", "p"]} for t in TOPICS])
print(ap_res.round(4).to_string(index=False))

fig, axes = plt.subplots(3, 1, figsize=(1.0, 2.7), sharex=True)
for t, ax in zip(TOPICS, axes):
    r = ap_res.set_index("topic").loc[t]
    star = "*" if r.P_spin < 0.05 else ""
    sh.plot_spin_null(ax, ap_null[t], r.rho_AP, TCOL[t], xlim=(-0.85, 0.85),
                      title=f"ρ = {r.rho_AP:.2f}{star} : {P_SPIN} = {sh.pstr(r.P_spin)}")
axes[-1].set_xlabel("Spearman ρ \n(vs A–P axis)", fontsize=6)
axes[-1].set_xticks([-0.5, 0, 0.5]); axes[-1].tick_params(labelsize=5)
fig.tight_layout(h_pad=0.6)
fig.savefig(f"{FIG_MAIN}/Fig2F_topic_AP_spin.pdf")
fig.savefig(f"{FIG_MAIN}/Fig2F_topic_AP_spin.png", dpi=300)
plt.show()

## 4) Map pattern tests

Two questions the A–P correlation alone does not answer, each its own supplementary figure.

### 4.1 Are the maps non-uniform?

A gene-set resampling test, not a spin test: is each map's spatial variance across the 34
cortical DK regions larger than for random size-matched gene sets (N = 20,000)? This asks
whether the topic has *any* regional structure, independently of whether that structure
aligns with a particular axis. Top row = the distribution of the 34 regional enrichment-Z
values; bottom row = the observed variance against the random-set null.

In [ ]:
nonuniform = {t: sh.map_nonuniformity(Z, info, GENE_SETS[t], n_perm=20000, seed=0) for t in TOPICS}
nonunif_tab = pd.DataFrame([
    {"topic": t, "n_genes": sum(g in Z.columns for g in GENE_SETS[t]),
     "var_enrichZ": v, "null_var_mean": nv.mean(),
     "p_nonuniform": (np.sum(nv >= v) + 1) / (len(nv) + 1)}
    for t, (_, v, nv) in nonuniform.items()])
print(nonunif_tab.round(4).to_string(index=False))

fig, axes = plt.subplots(2, 3, figsize=(7.09, 3.8))
for j, t in enumerate(TOPICS):
    obs_z, obs_var, null_var = nonuniform[t]
    p = (np.sum(null_var >= obs_var) + 1) / (len(null_var) + 1)

    a = axes[0, j]
    a.hist(obs_z, bins=np.arange(-5, 6.5, 0.5), color=TCOL[t], alpha=0.75, edgecolor="white", lw=0.3)
    a.axvline(0, color="0.6", lw=0.5, ls=":"); a.set_xlim(-5, 6.5)
    a.set_title(t, color=TCOL[t], fontsize=7, fontweight="bold")
    a.set_yticks([]); a.tick_params(labelsize=5)
    for s in ("top", "right", "left"): a.spines[s].set_visible(False)
    if j == 0: a.set_ylabel("# regions", fontsize=6)

    b = axes[1, j]
    b.hist(null_var, bins=50, color="0.85", edgecolor="black", linewidth=0.05, density=True)
    b.axvline(obs_var, color=TCOL[t], lw=2)
    b.set_title(f"observed var = {obs_var:.1f},  {P_PERM} = {sh.pstr(p)}", fontsize=6.5)
    b.set_yticks([]); b.tick_params(labelsize=5)
    for s in ("top", "right", "left"): b.spines[s].set_visible(False)
    if j == 0: b.set_ylabel("density", fontsize=6)
axes[0, 1].set_xlabel("regional enrichment Z", fontsize=6)
axes[1, 1].set_xlabel("spatial variance of enrichment Z", fontsize=6)
fig.suptitle("Regional enrichment-Z distribution (top) and its spatial variance vs "
             "random size-matched gene sets (N=20,000; bottom)", fontsize=7)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(f"{FIG_SUPP}/04_02_topic_nonuniformity.pdf")
fig.savefig(f"{FIG_SUPP}/04_02_topic_nonuniformity.png", dpi=300)
plt.show()

### 4.2 Do the three maps share one pattern?

Observed Spearman ρ between each pair of topic maps, tested against a null built by
spinning one map and correlating it with the other. The null pools both directions, so it
is symmetric in the pair. The spun topic maps are the cached focal nulls from the cluster
run, matched to topics by their (min, max) fingerprint — spinning permutes parcels but
preserves the value range, whereas the filename hash drifts if a map was re-resampled.

In [ ]:
focal = {}
for f in glob.glob(f"{CACHE}/focalnull_41k_n*_seed0_*.npy"):
    a = np.load(f)[:34]
    focal[(round(float(np.nanmin(a)), 2), round(float(np.nanmax(a)), 2))] = a
spun = {t: focal[(round(float(np.nanmin(topic_map[t])), 2),
                  round(float(np.nanmax(topic_map[t])), 2))] for t in TOPICS}

PAIRS = [("SYN", "GR"), ("GR", "MORPH"), ("MORPH", "SYN")]
sim_null, sim_rows = {}, []
for a, b in PAIRS:
    obs = sh.masked_spearman(topic_map[a], topic_map[b])
    null = np.concatenate([[sh.masked_spearman(spun[a][:, i], topic_map[b])
                            for i in range(spun[a].shape[1])],
                           [sh.masked_spearman(spun[b][:, i], topic_map[a])
                            for i in range(spun[b].shape[1])]])
    sim_null[(a, b)] = null
    sim_rows.append({"pair": f"{a}-{b}", "rho": obs, "P_spin": sh.emp_p(obs, null)})
sim_tab = pd.DataFrame(sim_rows)
print(sim_tab.round(4).to_string(index=False))

In [ ]:
from matplotlib.offsetbox import TextArea, HPacker, AnchoredOffsetbox

fig, axes = plt.subplots(1, 3, figsize=(7.09, 2.3), sharey=True)
for (a, b), ax in zip(PAIRS, axes):
    r = sim_tab.set_index("pair").loc[f"{a}-{b}"]
    sh.plot_spin_null(ax, sim_null[(a, b)], r.rho, "#b2182b", lw=2, xlim=(-1, 1),
                      title=f"ρ={r.rho:.2f}, {P_SPIN}={sh.pstr(r.P_spin)}")
    ax.title.set_fontsize(6.5)
    # pair label above the axis, each topic name in its own colour
    tp = lambda c: dict(color=c, fontsize=7, fontweight="bold")
    row = HPacker(children=[TextArea(a, textprops=tp(TCOL[a])),
                            TextArea(" – ", textprops=dict(color="black", fontsize=7)),
                            TextArea(b, textprops=tp(TCOL[b]))], align="baseline", pad=0, sep=0)
    ax.add_artist(AnchoredOffsetbox(loc="lower center", child=row, pad=0, frameon=False,
                                    bbox_to_anchor=(0.5, 1.10), bbox_transform=ax.transAxes,
                                    borderpad=0))
    ax.set_xlabel("Spearman ρ", fontsize=6); ax.tick_params(labelsize=5)
axes[0].set_ylabel("density", fontsize=6)
fig.suptitle("GO topic map similarity vs spin null (N=5,000)", fontsize=7, y=1.06)
fig.tight_layout()
fig.savefig(f"{FIG_SUPP}/04_02_topic_similarity_spin.pdf", bbox_inches="tight")
fig.savefig(f"{FIG_SUPP}/04_02_topic_similarity_spin.png", dpi=300, bbox_inches="tight")
plt.show()

with pd.ExcelWriter(f"{RESULTS}/04_02_topic_pattern_tests.xlsx") as xl:
    nonunif_tab.to_excel(xl, sheet_name="nonuniformity", index=False)
    ap_res.to_excel(xl, sheet_name="ap_pattern", index=False)
    sim_tab.to_excel(xl, sheet_name="pairwise_similarity", index=False)
print("saved ->", f"{RESULTS}/04_02_topic_pattern_tests.xlsx")

## 5) Occipital and gradient sensitivity

One supplementary figure covering both: **A** alignment with each gradient (A–P and Dear's
C1–C3), **B** the A–P gradient with and without the occipital regions, **C** the
gene-property-matched null, **D** drop-top-k. The methods are the same as notebook 04_01 §4,
applied per topic. The spun A–P axis is loaded from the cache notebook 04_01 wrote.

In [ ]:
N_SPIN_DROPOCC = 1000
ap, ap_spin_null = sh.cached_ap_spin_null(n_spin=N_SPIN_DROPOCC, density="41k", seed=0)

# gene covariates for the matched null: expression level, CDS length, constraint
tada = pd.read_csv(f"{PROJ}/data/full_results_wcounts_2025-10-08.txt", sep="\t")
cds = pd.read_csv(f"{PROJ}/data/gene_cds_length.csv")
gene_mean = expr.mean(axis=0)
covariates = (pd.DataFrame({"gene": background,
                            "expr": np.log10(gene_mean[background].values + 1e-6)})
              .merge(cds, left_on="gene", right_on="gene_name", how="left")
              .assign(cds=lambda d: np.log10(d.union_cds_length))
              .merge(tada[["gene", "LOEUF"]], on="gene", how="left")
              .dropna(subset=["expr", "cds", "LOEUF"]).set_index("gene"))
gene_occ = sh.occipital_gene_scores(Z, info)
chance = float(gene_occ.mean())

matched, dropocc, droptopk = {}, {}, {}
for t in TOPICS:
    matched[t] = sh.matched_null_occipital(GENE_SETS[t], gene_occ, background, covariates,
                                           n_perm=5000, seed=0)
    dropocc[t] = sh.dropocc_ap_test(topic_map[t], ap, ap_spin_null, dk)
    droptopk[t] = sh.drop_top_k(GENE_SETS[t], gene_occ, max_k=30)

pd.DataFrame([{"gene_set": t, "n": matched[t]["n"], "occipital_obs": matched[t]["obs"],
               "unmatched_mean": matched[t]["null_u"].mean(), "p_unmatched": matched[t]["p_u"],
               "matched_mean": matched[t]["null_m"].mean(), "p_matched": matched[t]["p_m"]}
              for t in TOPICS]).to_csv(f"{RESULTS}/04_02_topic_matched_null.csv", index=False)
pd.DataFrame([{"gene_set": t, **dropocc[t]} for t in TOPICS]) \
    .to_csv(f"{RESULTS}/04_02_topic_dropocc_ap_41k_n{N_SPIN_DROPOCC}.csv", index=False)

print(pd.DataFrame([{"topic": t, "matched_p": matched[t]["p_m"],
                     "rho_full": -dropocc[t]["rho_full"], "p_full": dropocc[t]["p_full"],
                     "rho_dropocc": -dropocc[t]["rho_dropocc"],
                     "p_dropocc": dropocc[t]["p_dropocc"]} for t in TOPICS])
      .round(3).to_string(index=False))

In [ ]:
from matplotlib.gridspec import GridSpec
from matplotlib.colors import to_rgba, LinearSegmentedColormap
from matplotlib.patches import Patch

tcmap = lambda t: LinearSegmentedColormap.from_list("t", ["#ffffff", TCOL[t]])  # white -> topic
GRADS, GLABS = ["AP", "C1", "C2", "C3"], ["A–P", "C1", "C2", "C3"]
# A-P is flipped to the posterior-positive convention used everywhere else; C1-C3 are not
signed = lambda t, g: (-1 if g == "AP" else 1) * float(spin_corr[t].loc[g, "enrich_z"])
vlim = max(abs(signed(t, g)) for t in TOPICS for g in GRADS)

xp = -ap
is_occ = np.array([r in sh.OCCIPITAL for r in dk])

fig = plt.figure(figsize=(11.5, 5.2))
outer = GridSpec(2, 1, figure=fig, hspace=0.5)
gtop = outer[0].subgridspec(1, 4, width_ratios=[1.4, 1, 1, 1], wspace=0.36)
gbot = outer[1].subgridspec(1, 4, width_ratios=[1, 1, 1, 1.4], wspace=0.36)

# --- A: gradient alignment. Bar height = |rho|, fill intensity = |rho|, sign printed
#        inside the bar, * = spin p < 0.05 ---
axA = fig.add_subplot(gtop[0, 0]); x = np.arange(4); w = 0.27
for j, t in enumerate(TOPICS):
    rr = np.array([signed(t, g) for g in GRADS])
    pp = np.array([float(spin_corr[t].loc[g, "p"]) for g in GRADS])
    axA.bar(x + (j - 1) * w, np.abs(rr), w, color=tcmap(t)(np.abs(rr) / vlim),
            edgecolor="#333", lw=0.4)
    for i, (h, ps, sr) in enumerate(zip(np.abs(rr), pp, rr)):
        axA.text(x[i] + (j - 1) * w, 0.02, "+" if sr > 0 else "−", ha="center", va="bottom",
                 fontsize=6, color="white" if (h / vlim) > 0.5 else "#333")
        if ps < 0.05:
            axA.text(x[i] + (j - 1) * w, h + 0.015, "*", ha="center", va="bottom",
                     fontsize=9, color=TCOL[t])
axA.set_xticks(x); axA.set_xticklabels(GLABS); axA.set_ylabel("|Spearman ρ|")
axA.set_title("Gradient alignment", fontsize=7); axA.set_ylim(0, 0.88)
axA.legend(handles=[Patch(color=TCOL[t]) for t in TOPICS], labels=TOPICS,
           fontsize=5, frameon=False, loc="upper right")
axA.text(0.02, 0.99, "fill intensity = |ρ|\n+/− sign · * spin p<0.05", transform=axA.transAxes,
         ha="left", va="top", fontsize=4.5, color="#555")
for s in ("top", "right"): axA.spines[s].set_visible(False)

# --- B: A-P scatter per topic, occipital regions marked and excluded from the solid fit ---
axB_first = None
for j, t in enumerate(TOPICS):
    ax = fig.add_subplot(gtop[0, 1 + j])
    if j == 0: axB_first = ax
    m = topic_map[t]
    ax.scatter(xp[~is_occ], m[~is_occ], s=13, color=TCOL[t], edgecolor="#333", lw=0.3, zorder=3)
    ax.scatter(xp[is_occ], m[is_occ], s=30, facecolor=to_rgba(TCOL[t], 0.3),
               edgecolor=TCOL[t], lw=0.8, ls="--", zorder=4)
    xs = np.linspace(xp.min(), xp.max(), 50)
    ax.plot(xs, np.polyval(np.polyfit(xp, m, 1), xs), "--", color="#aaa", lw=1)
    ax.plot(xs, np.polyval(np.polyfit(xp[~is_occ], m[~is_occ], 1), xs), "-", color=TCOL[t], lw=1.5)
    ax.set_xticks([]); ax.set_title(t, color=TCOL[t], fontweight="bold", fontsize=7)
    ax.text(0.04, 0.97, f"full ρ={-dropocc[t]['rho_full']:.2f} (p={dropocc[t]['p_full']:.3f})\n"
                        f"−occ ρ={-dropocc[t]['rho_dropocc']:.2f} (p={dropocc[t]['p_dropocc']:.3f})",
            transform=ax.transAxes, fontsize=4.5, va="top")
    if j == 0: ax.set_ylabel("enrichment Z")
    for s in ("top", "right"): ax.spines[s].set_visible(False)
fig.text(0.65, 0.505, r"anterior $\rightarrow$ posterior", ha="center", fontsize=7)

# --- C: gene-property-matched null per topic ---
axC_first = None
for j, t in enumerate(TOPICS):
    ax = fig.add_subplot(gbot[0, j])
    if j == 0: axC_first = ax
    r = matched[t]
    ax.hist(r["null_u"], bins=35, color="#cccccc", alpha=.85)
    ax.hist(r["null_m"], bins=35, color=to_rgba(TCOL[t], 0.75))
    ax.axvline(r["obs"], color="#b2182b", lw=1.5)
    ax.set_title(f"{t}  matched p={r['p_m']:.3f}\nunmatched p={r['p_u']:.4f}",
                 color=TCOL[t], fontsize=6)
    ax.set_xlabel("occipital enrichment")
    if j == 0: ax.set_ylabel("null count")
    for s in ("top", "right"): ax.spines[s].set_visible(False)

# --- D: drop-top-k ---
axD = fig.add_subplot(gbot[0, 3])
for t in TOPICS:
    ks, dtk, n = droptopk[t]
    axD.plot(ks, dtk, color=TCOL[t], lw=1.5, label=f"{t} (n={n})")
axD.axhline(chance, ls=":", color="#888", lw=1)
axD.text(0.99, chance, "chance", transform=axD.get_yaxis_transform(),
         ha="right", va="bottom", fontsize=5, color="#888")
axD.set_xlabel("top occipital genes removed (k)"); axD.set_ylabel("occipital enrichment (remaining)")
axD.set_title("Drop-top-k", fontsize=7); axD.legend(fontsize=5, frameon=False)
for s in ("top", "right"): axD.spines[s].set_visible(False)

for tag, ax in zip("ABCD", [axA, axB_first, axC_first, axD]):
    ax.text(-0.28, 1.14, tag, transform=ax.transAxes, fontsize=10, fontweight="bold", va="top")

fig.savefig(f"{FIG_SUPP}/04_02_topic_sensitivity.pdf", bbox_inches="tight")
fig.savefig(f"{FIG_SUPP}/04_02_topic_sensitivity.png", dpi=300, bbox_inches="tight")
plt.show()

## 6) Convergence with Gandal et al. 2022 cortical transcriptomic attenuation

Does the anterior-posterior topography of ASD rare-variant risk-gene enrichment converge
with the A–P gradient of transcriptomic *dysregulation* reported in Gandal et al. 2022
(Nature) — the attenuation of regional identity (ARI) that peaks in primary visual cortex?

Gandal et al. quantify ARI **pairwise**: for each of the C(11,2)=55 pairs of cortical
Brodmann regions, the number of genes differentially expressed *between* the two regions
is compared in ASD vs controls. Positive ⇒ fewer between-region DE genes in ASD ⇒ regional
identity collapsed. We summarise to a per-region score and Spearman-correlate it with our
per-region `z_vs_bg` across the 11 regions.

**Two attenuation metrics.** *Permutation* (`ARI_Gene_Permutation`, primary):
`CTL_NumDGE − ASD_NumDGE`, adjusted by the mean of the diagnosis-label permutation
distribution. *Bootstrap* (`ARI_Bootstrap2`, robustness): `Mean_CTL_P − Mean_ASD_P`, not
null-adjusted.

**Two summarisation schemes.** *anchor→PFC* (primary): each region's ARI against the
anterior pole, BA9 — the a-priori-correct reference for an anterior→posterior gradient,
and it removes the baseline-distinctiveness confound directly. *mean-over-pairs*
(anchor-free robustness): the mean of each region's 10 pairwise values. Anchoring to the
posterior pole (BA17/V1) instead inverts the sign, as expected for the opposite pole, and
is reported only for transparency.

**Hypothesis (a priori, directional):** risk-gene-enriched posterior regions show stronger
attenuation ⇒ **positive** ρ.

**Caveat:** n = 11 paired regions, so no spin test is possible and the analysis is
underpowered. ρ is reported with effect-size emphasis; p-values are nominal and
confirmatory of the pre-specified direction, not discovery.

### 6.1 DK → Brodmann crosswalk and the per-region attenuation score

In [ ]:
from scipy.stats import spearmanr, zscore, rankdata

GSETS = ["ASD"] + TOPICS
GSCOL = sh.GENE_SET_COLORS
FIG_G = f"{FIG_SUPP}/04_02_gandal2022"           # filename prefix, not a directory
GANDAL_S4 = f"{PROJ}/data/Gandal2022/TableS4.xlsx"

# per-region enrichment for all four gene sets (ASD map comes from notebook 01)
enrich = {"ASD": pd.read_csv(f"{RESULTS}/04_01_ASD_regional_enrichment.csv").set_index("region")["z_vs_bg"]}
for t in TOPICS:
    enrich[t] = topic_enr[t].set_index("region")["z_vs_bg"]

# DK -> Brodmann(11) crosswalk, from Gandal's AllenBrainAtlas_RegMatch sheet translated to
# DK names (PrG->precentral, Cun/LiG/OTG/SOG->occipital, HG->transversetemporal, TP->temporalpole).
# BA17 additionally includes pericalcarine, the DK striate label matching Gandal's -str structures.
DK2BA = {
    "BA9":          ["frontalpole"],
    "BA44_45":      ["parsopercularis", "parstriangularis"],
    "BA4_6":        ["precentral"],
    "BA24":         ["caudalanteriorcingulate", "rostralanteriorcingulate"],
    "BA17":         ["pericalcarine", "cuneus", "lingual", "lateraloccipital"],
    "BA39_40":      ["inferiorparietal", "supramarginal"],
    "BA3_1_2_5":    ["postcentral"],
    "BA7":          ["precuneus", "superiorparietal"],
    "BA20_37":      ["fusiform", "inferiortemporal"],
    "BA41_42_22":   ["transversetemporal", "superiortemporal"],
    "BA38":         ["temporalpole"],
}
missing = [d for dks in DK2BA.values() for d in dks if d not in enrich["ASD"].index]
assert not missing, f"DK labels absent from the enrichment maps: {missing}"
enr_ba = {g: pd.Series({b: np.mean([enrich[g][d] for d in dks]) for b, dks in DK2BA.items()})
          for g in GSETS}


def load_ari(sheet):
    d = pd.read_excel(GANDAL_S4, sheet_name=sheet)
    if sheet == "ARI_Gene_Permutation":
        d = d.rename(columns={d.columns[0]: "comp", d.columns[1]: "val"})   # null-adjusted
    else:
        d["val"] = d["Mean_CTL_P"] - d["Mean_ASD_P"]; d["comp"] = d["Comparison"]
    pair = d["comp"].str.split(" - ", expand=True).apply(lambda c: c.str.replace("-", "_").str.strip())
    d["A"], d["B"] = pair[0], pair[1]
    return d[["A", "B", "val"]]


METRICS = {"permutation": load_ari("ARI_Gene_Permutation"),
           "bootstrap":   load_ari("ARI_Bootstrap2")}
BAREG = sorted(set(METRICS["permutation"]["A"]) | set(METRICS["permutation"]["B"]))
assert len(BAREG) == 11


def per_region(d, anchor=None):
    """Collapse the pairwise ARI to one value per region: mean over its 10 pairs, or its
    single pair against `anchor` (which drops the anchor region itself)."""
    if anchor is None:
        return pd.Series({r: d.loc[(d.A == r) | (d.B == r), "val"].mean() for r in BAREG})
    out = {}
    for r in BAREG:
        if r == anchor:
            continue
        m = d[((d.A == r) & (d.B == anchor)) | ((d.A == anchor) & (d.B == r))]
        out[r] = m["val"].iloc[0] if len(m) else np.nan
    return pd.Series(out)


attn = {m: per_region(d) for m, d in METRICS.items()}
print("Per-region attenuation, mean-over-pairs (higher = more ASD attenuation):\n")
print(pd.DataFrame(attn).round(0).sort_values("permutation", ascending=False).to_string())
print(f"\nReproduces Gandal's V1/BA17 peak under both metrics: "
      f"{attn['permutation'].idxmax() == 'BA17' and attn['bootstrap'].idxmax() == 'BA17'}")

### 6.2 Convergence: risk-gene enrichment vs attenuation

In [ ]:
SCHEMES = {"anchor->BA9(PFC)": "BA9", "mean-over-pairs": None, "anchor->BA17(V1)": "BA17"}
rows = []
for mname, d in METRICS.items():
    for sname, anchor in SCHEMES.items():
        a = per_region(d, anchor)
        reg = [r for r in BAREG if r in a.index and not np.isnan(a[r])]
        for g in GSETS:
            rho, p = spearmanr(enr_ba[g][reg].values, a[reg].values)
            rows.append(dict(metric=mname, scheme=sname, gene_set=g,
                             n_regions=len(reg), rho=rho, p_nominal=p))
res = pd.DataFrame(rows)
res.to_csv(f"{RESULTS}/04_02_gandal2022_attenuation_correlation.csv", index=False)
print("Spearman ρ:")
print(res.pivot_table(index=["metric", "scheme"], columns="gene_set", values="rho")
         .reindex(list(SCHEMES), level=1).round(3)[GSETS].to_string())
print("\nNominal p:")
print(res.pivot_table(index=["metric", "scheme"], columns="gene_set", values="p_nominal")
         .reindex(list(SCHEMES), level=1).round(3)[GSETS].to_string())

In [ ]:
def scatter_grid(target, title, fname, xlab="ASD attenuation (per region)"):
    """One panel per gene set: per-region enrichment vs the Gandal target, ρ and nominal p."""
    reg = [r for r in BAREG if r in target.index and not np.isnan(target[r])]
    fig, axes = plt.subplots(2, 2, figsize=(5.2, 5.0))
    for ax, g in zip(axes.ravel(), GSETS):
        e, a = enr_ba[g][reg], target[reg]
        rho, p = spearmanr(e.values, a.values)
        ax.scatter(a.values, e.values, s=22, color=GSCOL[g], edgecolor="white", lw=.5, zorder=3)
        for r in reg:                                     # label V1 and the extremes only
            if r == "BA17" or e[r] == e.max() or e[r] == e.min():
                ax.annotate(r, (a[r], e[r]), fontsize=5, ha="left", va="bottom",
                            xytext=(2, 2), textcoords="offset points")
        ax.set_title(f"{g}   ρ={rho:+.2f}, p={p:.2f}")
        ax.set_xlabel(xlab); ax.set_ylabel("Risk-gene enrichment (z)")
        ax.axhline(0, color="0.8", lw=.6, zorder=0)
    fig.suptitle(title, fontsize=8)
    fig.tight_layout(rect=[0, 0, 1, 0.94]); fig.savefig(fname); plt.show()


for mname in ["permutation", "bootstrap"]:           # primary: anchored to the anterior pole
    scatter_grid(per_region(METRICS[mname], "BA9"),
                 f"Risk-gene enrichment vs A-P attenuation — {mname.upper()} metric\n"
                 "(10 regions, anchored to PFC/BA9; Spearman)",
                 f"{FIG_G}_enrichment_vs_attenuation_{mname}_anchorPFC.pdf")

for mname in ["permutation", "bootstrap"]:           # anchor-free robustness
    scatter_grid(attn[mname],
                 f"Risk-gene enrichment vs A-P attenuation — {mname.upper()} metric\n"
                 "(11 regions, mean-over-pairs; Spearman)",
                 f"{FIG_G}_enrichment_vs_attenuation_{mname}_meanpairs.pdf")

### 6.3 Robustness: per-region ASD DGE magnitude (Supplementary Data 3)

The pairwise ARI statistic forces a summarisation choice. Supplementary Data 3 gives a
genuine **per-region** ASD-vs-control differential-expression result for each of the 11
Brodmann regions — all 11 regions, no anchor, no pairwise collapse.

Choice of magnitude matters. Per-region sample sizes differ, so raw `# DEGs (FDR<0.05)` is
confounded by power. A pure `# |log2FC|>1` count over-corrects the other way — fold-change
estimates are noisiest in low-N regions, so spurious large FCs inflate the count there
(it ranks the lowest-power regions as most dysregulated, which is not credible). The
headline metric is therefore **`# genes with |log2FC|>1 AND FDR<0.05`**: an effect-size
threshold that also requires the effect to be real, robust to both confounds. The other
two are reported alongside for transparency.

> `TableS3.xlsx` is 57 MB and gitignored — download it from the Gandal 2022 supplement
> into `../data/Gandal2022/` to re-run this section.

In [ ]:
s3 = pd.read_excel(f"{PROJ}/data/Gandal2022/TableS3.xlsx", sheet_name="DEGene_Statistics",
                   usecols=[f"ASD_{b}_FDR" for b in BAREG] + [f"ASD_{b}_logFC" for b in BAREG])
nDEG    = pd.Series({b: int((s3[f"ASD_{b}_FDR"] < 0.05).sum()) for b in BAREG})
nLFC    = pd.Series({b: int((s3[f"ASD_{b}_logFC"].abs() > 1).sum()) for b in BAREG})
nSigLFC = pd.Series({b: int(((s3[f"ASD_{b}_logFC"].abs() > 1) & (s3[f"ASD_{b}_FDR"] < 0.05)).sum())
                     for b in BAREG})

print("Per-region ASD dysregulation magnitude:")
print(pd.DataFrame({"nDEG(FDR<.05)": nDEG, "nLFC(|lFC|>1 only)": nLFC,
                    "nSigLFC(|lFC|>1&FDR<.05)": nSigLFC})
      .sort_values("nSigLFC(|lFC|>1&FDR<.05)", ascending=False).to_string())
print(f"\nBoth headline metrics peak at V1/BA17: "
      f"{nDEG.idxmax() == 'BA17' and nSigLFC.idxmax() == 'BA17'}")

rows = []
for g in GSETS:
    for name, stat in [("nDEG_FDR05", nDEG), ("nLFC_lFC1_only", nLFC),
                       ("nSigLFC_lFC1_FDR05", nSigLFC)]:
        rho, p = spearmanr(enr_ba[g][BAREG].values, stat[BAREG].values)
        rows.append(dict(gene_set=g, metric=name, rho=rho, p_nominal=p, n_regions=11))
res3 = pd.DataFrame(rows)
res3.to_csv(f"{RESULTS}/04_02_gandal2022_S3_nDEG_correlation.csv", index=False)
print("\nSpearman(enrichment, per-region magnitude), all 11 regions:")
print(res3.pivot_table(index="metric", columns="gene_set", values="rho")
          .reindex(["nDEG_FDR05", "nLFC_lFC1_only", "nSigLFC_lFC1_FDR05"]).round(3)[GSETS].to_string())

In [ ]:
scatter_grid(nSigLFC,
             "Risk-gene enrichment vs per-region ASD dysregulation\n"
             "(# genes |log2FC|>1 & FDR<0.05; 11 regions, Supplementary Data 3; Spearman)",
             f"{FIG_G}_enrichment_vs_nSigLFC_S3.pdf",
             xlab="# ASD DEGs (|log2FC|>1 & FDR<0.05)")
scatter_grid(nDEG,
             "Comparison: raw # DEGs (FDR<0.05) — sample-size-confounded\n"
             "(11 regions, Supplementary Data 3; Spearman)",
             f"{FIG_G}_enrichment_vs_nDEG_S3.pdf", xlab="# ASD DEGs (FDR<0.05)")
scatter_grid(nLFC,
             "Comparison: # genes |log2FC|>1 only (no FDR filter) — noise-dominated\n"
             "(11 regions, Supplementary Data 3; Spearman)",
             f"{FIG_G}_enrichment_vs_nLFC_S3.pdf", xlab="# ASD DEGs (|log2FC|>1, no FDR)")

### 6.4 Permutation p-values (gene-set resampling null)

With n = 11 regions the nominal Spearman p assumes an asymptotic null that does not really
hold. A better test asks: is *our* gene set's enrichment map more correlated with the
Gandal target than a random size-matched brain-expressed gene set's map? For each of
10,000 random sets the enrichment map is built **standardised per DK region across the null
draws, then averaged to the 11 BA regions — exactly the `z_vs_bg` construction** — so the
observed ρ here equals the §6.2–6.3 Spearman values (asserted below).

The per-region standardisation is essential: on a *raw* expression map random sets share a
systematic spatial bias that inflates significance (~p=0.003), the exact confound
`z_vs_bg` removes. Standardised, the null mean is ≈0 and the p-values are honest. Reported
one-sided (matching the a-priori positive direction), with two-sided alongside.

In [ ]:
GENE_SETS_ALL = {"ASD": tada[(tada.FDR < 0.001) & (~tada.FLAG)]["gene"].tolist(), **GENE_SETS}

Zarr = zscore(expr[background].to_numpy(), axis=0)
gene2col = {g: i for i, g in enumerate(background)}
region_label = np.array([info.loc[i, "label"] for i in expr.index])
DKS = [d for b in BAREG for d in DK2BA[b]]

A_dk = np.zeros((len(DKS), Zarr.shape[0]))              # AHBA rows -> DK label (mean)
for di, d in enumerate(DKS):
    rr = np.where(region_label == d)[0]
    if len(rr):
        A_dk[di, rr] = 1.0 / len(rr)
B_ba = np.zeros((len(BAREG), len(DKS)))                 # DK -> BA (mean), applied AFTER standardising
for bi, b in enumerate(BAREG):
    for d in DK2BA[b]:
        B_ba[bi, DKS.index(d)] = 1.0 / len(DK2BA[b])

N_PERM, SEED = 10000, 0
TARGETS = {"ARI(perm, mean-pairs)": attn["permutation"], "S3 #DEG(FDR<.05)": nDEG,
           "S3 #DEG(|lFC|>1&FDR<.05)": nSigLFC, "S3 #DEG(|lFC|>1 only)": nLFC}
perm_null, rows = {}, []
for g in GSETS:
    idx = np.array([gene2col[x] for x in GENE_SETS_ALL[g] if x in gene2col])
    rng = np.random.default_rng(SEED)
    D_dk = np.array([A_dk @ Zarr[:, rng.choice(Zarr.shape[1], len(idx), replace=False)].mean(1)
                     for _ in range(N_PERM)])
    mu, sd = D_dk.mean(0), D_dk.std(0)                  # per-DK null moments = the z_vs_bg standardisation
    ba_obs = B_ba @ ((A_dk @ Zarr[:, idx].mean(1) - mu) / sd)
    ba_null = ((D_dk - mu) / sd) @ B_ba.T
    for tname, tv in TARGETS.items():
        tr = rankdata(tv[BAREG].values)
        obs = np.corrcoef(rankdata(ba_obs), tr)[0, 1]
        null = np.array([np.corrcoef(rankdata(ba_null[i]), tr)[0, 1] for i in range(N_PERM)])
        perm_null[(g, tname)] = (obs, null)
        rows.append(dict(gene_set=g, target=tname, n_genes=len(idx), rho=obs,
                         null_mean=null.mean(),
                         p_perm_1sided=(np.sum(null >= obs) + 1) / (N_PERM + 1),
                         p_perm_2sided=(np.sum(np.abs(null) >= abs(obs)) + 1) / (N_PERM + 1)))
permres = pd.DataFrame(rows)
permres.to_csv(f"{RESULTS}/04_02_gandal2022_permutation_pvalues.csv", index=False)

# the observed rho must reproduce the §6.2-6.3 Spearman exactly (same map construction)
assert abs(spearmanr(enr_ba["ASD"][BAREG].values, nDEG[BAREG].values)[0]
           - permres.query("gene_set=='ASD' and target=='S3 #DEG(FDR<.05)'")["rho"].iloc[0]) < 1e-9
print(f"gene-set resampling null, N={N_PERM}:")
print(permres.round(4).to_string(index=False))

In [ ]:
for tgt, fname, suptitle in [
    ("S3 #DEG(|lFC|>1&FDR<.05)", f"{FIG_G}_permutation_null_S3nSigLFC.pdf",
     "Gene-set resampling null vs per-region ASD dysregulation "
     "(# genes |log2FC|>1 & FDR<0.05, Supp. Data 3)"),
    ("S3 #DEG(|lFC|>1 only)", f"{FIG_G}_permutation_null_S3nLFC.pdf",
     "Non-robust metric: pure # genes |log2FC|>1 (no FDR filter) — observed ρ falls inside the null"),
]:
    fig, axes = plt.subplots(1, 4, figsize=(9, 2.3))
    for ax, g in zip(axes, GSETS):
        obs, null = perm_null[(g, tgt)]
        p1 = (np.sum(null >= obs) + 1) / (N_PERM + 1)
        ax.hist(null, bins=40, color="0.82", edgecolor="none")
        ax.axvline(obs, color=GSCOL[g], lw=2); ax.axvline(0, color="0.5", lw=.6, ls=":")
        ax.set_title(f"{g}\nρ={obs:+.2f}, p={p1:.3f}", fontsize=7)
        ax.set_xlabel("null ρ (random gene sets)"); ax.set_yticks([])
    fig.suptitle(suptitle, fontsize=8)
    fig.tight_layout(rect=[0, 0, 1, 0.9]); fig.savefig(fname); plt.show()

### 6.5 Drop-V1 sensitivity

V1/BA17 is the peak of both maps — highest risk-gene enrichment and highest dysregulation —
so it is the strongest single concordant point. Is the correlation *only* V1? Each
correlation is recomputed over the remaining 10 regions with BA17 removed.

In [ ]:
BA10 = [b for b in BAREG if b != "BA17"]
DROP_METRICS = {"ARI perm (mean-pairs)": attn["permutation"], "nDEG (FDR<.05)": nDEG,
                "nSigLFC (|lFC|>1&FDR<.05)": nSigLFC}
rows = []
for mname, mv in DROP_METRICS.items():
    for g in GSETS:
        rho_all = spearmanr(enr_ba[g][BAREG].values, mv[BAREG].values)[0]
        rho_drop, p_drop = spearmanr(enr_ba[g][BA10].values, mv[BA10].values)
        rows.append(dict(metric=mname, gene_set=g, rho_all11=rho_all,
                         rho_dropV1=rho_drop, p_dropV1=p_drop))
dv = pd.DataFrame(rows)
dv.to_csv(f"{RESULTS}/04_02_gandal2022_dropV1_sensitivity.csv", index=False)
for mname in DROP_METRICS:
    print(f"--- {mname} ---")
    print(dv[dv.metric == mname].set_index("gene_set")[["rho_all11", "rho_dropV1", "p_dropV1"]]
          .round(3).to_string(), "\n")

### 6.6 Summary

- Gandal et al.'s central result reproduces here: transcriptomic regional-identity
  attenuation in ASD peaks in **BA17/V1** under both metrics.
- ASD risk-gene enrichment is **directionally concordant** with that attenuation gradient
  across the 11 regions (positive ρ), driven by the **GR** topic — consistent with the
  posterior/occipital A–P enrichment GR carries in §3. Primary (anchor→PFC): ASD ρ≈0.44–0.54,
  GR ρ≈0.43–0.53 across the two metrics.
- **Robust to summarisation and to metric:** the anterior-pole anchor and the anchor-free
  mean-over-pairs agree, and the sample-size-robust per-region statistic gives GR ρ=+0.54,
  ASD ρ=+0.42. Only a pure |log2FC|>1 count dissents, and it is noise-dominated in low-N regions.
- **Permutation p-values confirm rather than strengthen:** GR p≈0.08–0.15, ASD p≈0.14–0.23
  (one-sided). A raw-map null would report p≈0.003, but that exploits generic expression
  topography and is not the honest test.
- **Partly, not solely, V1:** dropping BA17 lowers the DEG-based correlations but leaves a
  moderate GR positive (nSigLFC 0.54→0.38); the pairwise ARI is more V1-dependent (0.36→0.15).
- **Bottom line:** with n=11 regions this is a *directional* convergence, underpowered and
  not significant. n=11 is the fundamental limit; no null choice overcomes it.